# Test: Prior-Guided Full-Dimensional Model (Textures, 256-PC)

Interactive validation notebook for the new prior-guided drift-diffusion pipeline
before launching the full SLURM grid search.

This notebook mirrors:
- `test_3step_grid_search.ipynb` for real data/encoder setup
- `2d_guided_drift_sandbox.ipynb` for prior-guided structure

**Coverage:**
1. Setup & imports (including `cox` mock)
2. Load auditory-texture experiment data + encode to 256 PCs
3. Load learned `ScoreFunction` and sanity-check score outputs
4. Generate high-diversity multi-ISI sequences
5. Single default-parameter run + per-ISI d′
6. Small Monte Carlo aggregation (N=3)
7. Quick η sweep (`eta ∈ {0.0, 0.05, 0.2}`)


In [1]:
import os, sys, types
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Mock 'cox' and 'cox.store' so constants.py imports don't crash
cox_mock = types.ModuleType('cox')
store_mock = types.ModuleType('cox.store')
store_mock.PYTORCH_STATE = 'pytorch_state'
cox_mock.store = store_mock
sys.modules['cox'] = cox_mock
sys.modules['cox.store'] = store_mock

# Local repo first
sys.path.insert(0, os.path.abspath('..'))

# Cluster paths (needed for encoder + score model stack)
sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set

from utls.encoders import *
from utls.runners_prior import run_model_core_prior
from utls.runners_utils import load_experiment_data, build_encoder, encode_stimuli
from utls.toy_experiments import make_high_diversity_sequences
from utls.roc_utils import roc_from_arrays
from utls.analysis_helpers import auroc_to_dprime
from src.model.ScoreFunction import ScoreFunction

%matplotlib inline
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

print(f'torch={torch.__version__}')
print('Imports OK')


torch=2.3.0
Imports OK


/orcd/data/jhm/001/om2/lakshmin/audio-prior/utils/sde_utils.py:10: SyntaxWarning: invalid escape sequence '\s'
  """Compute the mean and standard deviation of $p_{0t}(x(t) | x(0))$.
/orcd/data/jhm/001/om2/lakshmin/audio-prior/utils/sde_utils.py:24: SyntaxWarning: invalid escape sequence '\s'
  """Compute the diffusion coefficient of our SDE.


## 1) Load auditory-texture experiment data & encode stimuli (256 PCs)

In [2]:
# Task 2 = auditory textures (for eventual human comparison)
which_task = 2
is_multi = True

exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = load_experiment_data(which_task=which_task, which_isi=None, is_multi=is_multi)

print(f'Task: {task_name} ({hr_task_name})')
print(f'N sequences from experiment loader: {len(exp_list)}')
print(f'N unique stimuli: {len(all_files)}')


Task: atexts (Auditory Textures)
N sequences from experiment loader: 120
N unique stimuli: 80


In [ ]:
# Build texture PCA encoder (256 PCs)
encoder_cfg = dict(
    encoder_type='texture_pca',
    model_name='texture_stats_to_pc',  # used for cluster model-dir path print
    task='auditory_textures',
    statistics_dict=statistics_set.statistics,
    model_params=model_params,
    pc_dims=256,
    sr=20000,
    duration=2.0,
    rms_level=0.05,
    device='cuda',
)

print('Building encoder: texture_pca (256 PCs) ...')
encoder = build_encoder(encoder_cfg)

print(f'Encoding {len(all_files)} stimuli ...')
X0 = encode_stimuli(encoder, all_files)
X0 = X0.float().to('cuda')

print(f'X0.shape = {tuple(X0.shape)}')
print(f'Expected second dim = 256 -> {X0.shape[1]}')
print(f'Finite check: {torch.isfinite(X0).all().item()}')
print(f'Any NaN: {torch.isnan(X0).any().item()}')
print(f'Mean ± std: {X0.mean().item():.4f} ± {X0.std().item():.4f}')


## 2) Load prior score model and sanity-check outputs

In [ ]:
device = 'cuda'
score_model = ScoreFunction(mode='textures', restart=True, device=device)

# Quick sanity check on a few encoded stimuli
idx = torch.randperm(X0.shape[0], device=X0.device)[:8]
x_test = X0[idx]
with torch.no_grad():
    s_test = score_model.forward(x_test)

# ScoreFunction typically returns [B,1,1,D] for [B,D] input
if s_test.dim() == 4:
    s_flat = s_test.reshape(s_test.shape[0], -1)
elif s_test.dim() == 2:
    s_flat = s_test
else:
    raise ValueError(f'Unexpected score output shape: {tuple(s_test.shape)}')

norms = torch.norm(s_flat, dim=1)
print(f'score output shape: {tuple(s_test.shape)}')
print(f'score norms (first 8): {norms.detach().cpu().numpy()}')
print(f'All finite: {torch.isfinite(s_flat).all().item()}')
print(f'Any non-zero norm: {(norms > 0).any().item()}')


## 3) Generate high-diversity multi-ISI sequences

In [ ]:
ISI_VALUES = [0, 1, 2, 4, 8, 16, 32, 64]
N_SEQUENCES = 30
SEQ_LENGTH = 120
MIN_PAIRS_PER_ISI = 4
SEED = 43

stimulus_pool = sorted({s for seq in exp_list for s in seq})
print(f'Stimulus pool size: {len(stimulus_pool)}')

experiment_list, isi_keys = make_high_diversity_sequences(
    stimulus_pool=stimulus_pool,
    isi_values=ISI_VALUES,
    n_sequences=N_SEQUENCES,
    length=SEQ_LENGTH,
    min_pairs_per_isi=MIN_PAIRS_PER_ISI,
    seed=SEED,
)

print(f'Generated {len(experiment_list)} sequences; sequence length={len(experiment_list[0])}')
print(f'ISI keys returned: {sorted(list(isi_keys.keys()))}')

isi_counts = {isi: len(isi_keys.get(isi, [])) for isi in ISI_VALUES}
print('Pair counts by ISI:', isi_counts)


## 4) Single run with default parameters

In [ ]:
# Defaults requested
sigma0 = 5.0
sigma = 1.0
eta = 0.1
metric = 'cosine'

run_single = run_model_core_prior(
    sigma0=sigma0,
    sigma=sigma,
    X0=X0,
    name_to_idx=name_to_idx,
    experiment_list=experiment_list,
    score_model=score_model,
    drift_step_size=eta,
    metric=metric,
    seed=0,
)

print(f"N hit scores: {len(run_single['hits'])}")
print(f"N FA scores: {len(run_single['fas'])}")
print(f"T_max: {run_single['T_max']}")

# Build per-ISI summary + d'
single_rows = []
for isi in ISI_VALUES:
    key = isi + 1  # runner stores ISI as intervening+1
    hits_raw = run_single['isi_hit_dists'].get(key, [])
    hit_scores = np.array([s for s, _ in hits_raw], dtype=float)
    fa_scores = np.array(run_single['fas'], dtype=float)

    if len(hit_scores) > 2 and len(fa_scores) > 2:
        roc = roc_from_arrays(hit_scores, fa_scores, score_type=run_single['score_type'])
        dprime = auroc_to_dprime(roc[2]) if roc is not None else np.nan
    else:
        dprime = np.nan

    single_rows.append({
        'isi': isi,
        'n_hits': len(hit_scores),
        'n_fas_global': len(fa_scores),
        'hit_mean': np.mean(hit_scores) if len(hit_scores) else np.nan,
        'hit_std': np.std(hit_scores) if len(hit_scores) else np.nan,
        'fa_mean_global': np.mean(fa_scores) if len(fa_scores) else np.nan,
        'fa_std_global': np.std(fa_scores) if len(fa_scores) else np.nan,
        "dprime": dprime,
    })

single_df = pd.DataFrame(single_rows)
display(single_df)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(single_df['isi'], single_df['dprime'], marker='o', lw=2)
ax.set_xscale('log', base=2)
ax.set_xticks(ISI_VALUES)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel('ISI')
ax.set_ylabel("d'")
ax.set_title("Single-run d' vs ISI (default prior-guided params)")
ax.grid(alpha=0.25)
plt.show()


## 5) Small Monte Carlo aggregation (N_MC = 3)

In [ ]:
N_MC = 3

all_isi_hits = defaultdict(list)
all_fas = []

for rep in range(N_MC):
    run_rep = run_model_core_prior(
        sigma0=sigma0,
        sigma=sigma,
        X0=X0,
        name_to_idx=name_to_idx,
        experiment_list=experiment_list,
        score_model=score_model,
        drift_step_size=eta,
        metric=metric,
        seed=1000 + rep,
    )
    for isi in ISI_VALUES:
        all_isi_hits[isi].extend(run_rep['isi_hit_dists'].get(isi + 1, []))
    all_fas.extend(run_rep['fas'])

fas_arr = np.array(all_fas, dtype=float)

mc_rows = []
for isi in ISI_VALUES:
    hits_scores = np.array([s for s, _ in all_isi_hits[isi]], dtype=float)

    if len(hits_scores) > 2 and len(fas_arr) > 2:
        roc = roc_from_arrays(hits_scores, fas_arr, score_type='distance')
        dprime = auroc_to_dprime(roc[2]) if roc is not None else np.nan
    else:
        dprime = np.nan

    mc_rows.append({'isi': isi, 'n_hits': len(hits_scores), 'n_fas': len(fas_arr), "dprime": dprime})

mc_df = pd.DataFrame(mc_rows)
display(mc_df)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mc_df['isi'], mc_df['dprime'], marker='o', lw=2, label=f'N_MC={N_MC}')
ax.set_xscale('log', base=2)
ax.set_xticks(ISI_VALUES)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel('ISI')
ax.set_ylabel("d'")
ax.set_title("Monte Carlo aggregated d' vs ISI")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

# quick trend diagnostic (not strict pass/fail)
vals = mc_df['dprime'].to_numpy(dtype=float)
mask = np.isfinite(vals)
if mask.sum() > 2:
    slope = np.polyfit(np.log2(np.array(ISI_VALUES)[mask]), vals[mask], 1)[0]
    print(f"log2-ISI slope of d' = {slope:.4f} (expected negative)")


## 6) Quick η sweep (drift effect)
Compare `eta ∈ {0.0, 0.05, 0.2}` with fixed `(sigma0, sigma)`.


In [ ]:
eta_grid = [0.0, 0.05, 0.2]
eta_curves = {}

for eta_i in eta_grid:
    all_isi_hits = defaultdict(list)
    all_fas = []

    for rep in range(N_MC):
        run_rep = run_model_core_prior(
            sigma0=sigma0,
            sigma=sigma,
            X0=X0,
            name_to_idx=name_to_idx,
            experiment_list=experiment_list,
            score_model=score_model,
            drift_step_size=eta_i,
            metric=metric,
            seed=3000 + rep,
        )
        for isi in ISI_VALUES:
            all_isi_hits[isi].extend(run_rep['isi_hit_dists'].get(isi + 1, []))
        all_fas.extend(run_rep['fas'])

    fas_arr = np.array(all_fas, dtype=float)
    curve = []
    for isi in ISI_VALUES:
        hits_scores = np.array([s for s, _ in all_isi_hits[isi]], dtype=float)
        if len(hits_scores) > 2 and len(fas_arr) > 2:
            roc = roc_from_arrays(hits_scores, fas_arr, score_type='distance')
            curve.append(auroc_to_dprime(roc[2]) if roc is not None else np.nan)
        else:
            curve.append(np.nan)
    eta_curves[eta_i] = np.array(curve, dtype=float)

eta_df = pd.DataFrame({'isi': ISI_VALUES, **{f'eta={e}': eta_curves[e] for e in eta_grid}})
display(eta_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

# Left: overlay curves
for eta_i in eta_grid:
    axes[0].plot(ISI_VALUES, eta_curves[eta_i], marker='o', lw=2, label=f'eta={eta_i}')
axes[0].set_xscale('log', base=2)
axes[0].set_xticks(ISI_VALUES)
axes[0].get_xaxis().set_major_formatter(plt.ScalarFormatter())
axes[0].set_xlabel('ISI')
axes[0].set_ylabel("d'")
axes[0].set_title("d' vs ISI by drift step size η")
axes[0].grid(alpha=0.25)
axes[0].legend()

# Right: delta-to-eta0 view
base = eta_curves[0.0]
for eta_i in eta_grid[1:]:
    axes[1].plot(ISI_VALUES, eta_curves[eta_i] - base, marker='o', lw=2, label=f'eta={eta_i} - eta=0')
axes[1].axhline(0, color='k', lw=1, alpha=0.6)
axes[1].set_xscale('log', base=2)
axes[1].set_xticks(ISI_VALUES)
axes[1].get_xaxis().set_major_formatter(plt.ScalarFormatter())
axes[1].set_xlabel('ISI')
axes[1].set_ylabel("Δd' vs eta=0")
axes[1].set_title('Drift effect (difference curves)')
axes[1].grid(alpha=0.25)
axes[1].legend()

plt.show()

# Simple equality check to confirm eta>0 changes predictions
for eta_i in eta_grid[1:]:
    same = np.allclose(eta_curves[eta_i], eta_curves[0.0], equal_nan=True)
    print(f'eta={eta_i} identical to eta=0? -> {same}')


## 7) Verification checklist

- [ ] `X0.shape == [N, 256]`
- [ ] Score outputs are finite and non-zero
- [ ] d′ values are finite for most ISIs
- [ ] d′ tends to decrease with ISI (expected pattern)
- [ ] `eta > 0` changes d′ curves vs `eta = 0`

If all checks look sensible, you can launch the full SLURM grid search via:

```bash
python src/model/run_prior_guided_grid_search.py --job-index 0 --n-mc 10
```
